In [1]:
# ============================================================
# PHARMAOPT — NOTEBOOK 07
# CMS PART D DEMAND / UTILIZATION DATA
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd()

print("PharmaOpt project folder:")
print(PROJECT_DIR)

print("\nNotebook 07 ready.")

PharmaOpt project folder:
C:\Users\jpiza\PharmaOpt

Notebook 07 ready.


In [2]:
# ============================================================
# CHECK CMS DOWNLOADS
# ============================================================

from pathlib import Path

PROJECT_DIR = Path.cwd()

print("FILES NOW IN PHARMAOPT")
print("=" * 70)

for file in PROJECT_DIR.iterdir():
    if file.is_file():
        print(file.name)

FILES NOW IN PHARMAOPT
01_fda_shortage_audit.ipynb.ipynb
02_fda_ndc_audit.ipynb.ipynb
03_fda_recall_audit.ipynb.ipynb
04_fda_data_integration.ipynb.ipynb
05_mysql_data_load.ipynb.ipynb
06_pharmaopt_analytics.ipynb.ipynb
07_cms_partd_utilization.ipynb.ipynb
drug-enforcement-0001-of-0001.json.zip
drug-ndc-0001-of-0001.json.zip
drug-shortages-0001-of-0001.json.zip


In [3]:
# ============================================================
# FIND RECENT CMS DOWNLOADS
# ============================================================

from pathlib import Path
from datetime import datetime

downloads = Path.home() / "Downloads"

print("SEARCHING:")
print(downloads)
print("\nRECENT DATA / CMS-LIKE FILES")
print("=" * 90)

extensions = {".csv", ".zip", ".xlsx", ".parquet", ".pdf"}

files = [
    f for f in downloads.rglob("*")
    if f.is_file()
    and f.suffix.lower() in extensions
]

# Most recently downloaded first
files = sorted(
    files,
    key=lambda x: x.stat().st_mtime,
    reverse=True
)

for f in files[:30]:
    modified = datetime.fromtimestamp(
        f.stat().st_mtime
    ).strftime("%Y-%m-%d %H:%M:%S")

    print(
        f"{modified} | "
        f"{f.name}"
    )

SEARCHING:
C:\Users\jpiza\Downloads

RECENT DATA / CMS-LIKE FILES
2026-09-28 22:26:33 | Medicare Part D Prescribers - by Geography and Drug (1).zip
2026-09-28 22:26:16 | Medicare Part D Prescribers - by Geography and Drug.zip
2026-09-25 22:09:32 | drug-ndc-0001-of-0001.json.zip
2026-09-25 22:09:31 | drug-shortages-0001-of-0001.json.zip
2026-09-25 22:09:27 | drug-enforcement-0001-of-0001.json.zip
2026-09-25 00:00:12 | drug-shortages-0001-of-0001.json (1).zip
2026-09-24 20:12:24 | Module 2 Slides - Part 1 (1).zip
2026-09-24 11:04:03 | Module 2 Slides - Part 2.zip
2026-09-24 11:03:57 | Module 2 Slides - Part 1.zip
2026-09-17 13:00:24 | Q4_data (1).csv
2026-09-17 12:59:49 | Q3_data (2).csv
2026-09-15 13:49:55 | Chapter_3_Financial_Environment_Textbook.pdf
2026-09-15 12:32:09 | ~$Homework_1_solutions.xlsx
2026-09-15 12:31:34 | Homework_1_solutions.xlsx
2026-09-14 08:30:53 | ~$FTIS NTD Excel Example Dataset_with changes.xlsx
2026-09-14 08:30:43 | FTIS NTD Excel Example Dataset_with changes.x

In [4]:
# ============================================================
# INSPECT CMS ZIP DOWNLOADS
# ============================================================

from pathlib import Path
import zipfile

downloads = Path.home() / "Downloads"

cms_zips = [
    downloads / "Medicare Part D Prescribers - by Geography and Drug.zip",
    downloads / "Medicare Part D Prescribers - by Geography and Drug (1).zip"
]

for zip_path in cms_zips:

    print("\n" + "=" * 100)
    print("ZIP FILE:", zip_path.name)
    print("=" * 100)

    with zipfile.ZipFile(zip_path, "r") as z:

        contents = z.namelist()

        print("Files inside:", len(contents))
        print()

        for filename in contents:
            print(filename)


ZIP FILE: Medicare Part D Prescribers - by Geography and Drug.zip
Files inside: 9

Medicare Part D Prescribers - by Geography and Drug/
Medicare Part D Prescribers - by Geography and Drug/2024/
Medicare Part D Prescribers - by Geography and Drug/2024/MUP_DPR_RY26_P04_V10_DY24_Geo.csv
Medicare Part D Prescribers - by Geography and Drug/2022/
Medicare Part D Prescribers - by Geography and Drug/2022/MUP_DPR_RY24_P04_V10_DY22_Geo.csv
Medicare Part D Prescribers - by Geography and Drug/2021/
Medicare Part D Prescribers - by Geography and Drug/2021/MUP_DPR_RY23_P04_V10_DY21_Geo.csv
Medicare Part D Prescribers - by Geography and Drug/2023/
Medicare Part D Prescribers - by Geography and Drug/2023/MUP_DPR_RY25_P04_V10_DY23_Geo.csv

ZIP FILE: Medicare Part D Prescribers - by Geography and Drug (1).zip
Files inside: 3

Medicare Part D Prescribers - by Geography and Drug/
Medicare Part D Prescribers - by Geography and Drug/2024/
Medicare Part D Prescribers - by Geography and Drug/2024/MUP_DPR_RY2

In [5]:
# ============================================================
# PHARMAOPT — COPY RAW CMS FILES INTO PROJECT
# ============================================================

from pathlib import Path
import shutil

downloads = Path.home() / "Downloads"
project = Path.cwd()

cms_data_zip_source = (
    downloads /
    "Medicare Part D Prescribers - by Geography and Drug.zip"
)

cms_dictionary_zip_source = (
    downloads /
    "Medicare Part D Prescribers - by Geography and Drug (1).zip"
)

cms_data_zip = (
    project /
    "Medicare Part D Prescribers - by Geography and Drug.zip"
)

cms_dictionary_zip = (
    project /
    "Medicare Part D Prescribers - by Geography and Drug (1).zip"
)

shutil.copy2(
    cms_data_zip_source,
    cms_data_zip
)

shutil.copy2(
    cms_dictionary_zip_source,
    cms_dictionary_zip
)

print("CMS files copied into PharmaOpt.")
print()
print(cms_data_zip.name)
print(cms_dictionary_zip.name)

CMS files copied into PharmaOpt.

Medicare Part D Prescribers - by Geography and Drug.zip
Medicare Part D Prescribers - by Geography and Drug (1).zip


In [6]:
# ============================================================
# PHARMAOPT — LOAD CMS PART D DATA
# ============================================================

import pandas as pd
import zipfile

cms_members = {

    2021:
    "Medicare Part D Prescribers - by Geography and Drug/2021/"
    "MUP_DPR_RY23_P04_V10_DY21_Geo.csv",

    2022:
    "Medicare Part D Prescribers - by Geography and Drug/2022/"
    "MUP_DPR_RY24_P04_V10_DY22_Geo.csv",

    2023:
    "Medicare Part D Prescribers - by Geography and Drug/2023/"
    "MUP_DPR_RY25_P04_V10_DY23_Geo.csv",

    2024:
    "Medicare Part D Prescribers - by Geography and Drug/2024/"
    "MUP_DPR_RY26_P04_V10_DY24_Geo.csv"
}


cms_data = {}


with zipfile.ZipFile(cms_data_zip, "r") as z:

    for year, member in cms_members.items():

        print(f"Loading CMS {year}...")

        with z.open(member) as f:

            df = pd.read_csv(
                f,
                low_memory=False
            )

        df["data_year"] = year

        cms_data[year] = df

        print(
            f"  Rows: {len(df):,} | "
            f"Columns: {len(df.columns):,}"
        )


print("\nCMS DATA LOADED")
print("=" * 70)

for year, df in cms_data.items():

    print(
        year,
        "→",
        f"{len(df):,} rows"
    )

Loading CMS 2021...
  Rows: 114,891 | Columns: 23
Loading CMS 2022...
  Rows: 115,396 | Columns: 23
Loading CMS 2023...
  Rows: 115,936 | Columns: 23
Loading CMS 2024...
  Rows: 117,661 | Columns: 23

CMS DATA LOADED
2021 → 114,891 rows
2022 → 115,396 rows
2023 → 115,936 rows
2024 → 117,661 rows


In [7]:
# ============================================================
# CMS STRUCTURE AUDIT
# ============================================================

for year in sorted(cms_data):

    df = cms_data[year]

    print("\n" + "=" * 90)
    print(f"CMS PART D — {year}")
    print("=" * 90)

    print("\nCOLUMNS:")

    for column in df.columns:
        print(" -", column)

    print("\nFIRST 3 ROWS:")

    display(
        df.head(3)
    )


CMS PART D — 2021

COLUMNS:
 - Prscrbr_Geo_Lvl
 - Prscrbr_Geo_Cd
 - Prscrbr_Geo_Desc
 - Brnd_Name
 - Gnrc_Name
 - Tot_Prscrbrs
 - Tot_Clms
 - Tot_30day_Fills
 - Tot_Drug_Cst
 - Tot_Benes
 - GE65_Sprsn_Flag
 - GE65_Tot_Clms
 - GE65_Tot_30day_Fills
 - GE65_Tot_Drug_Cst
 - GE65_Bene_Sprsn_Flag
 - GE65_Tot_Benes
 - LIS_Bene_Cst_Shr
 - NonLIS_Bene_Cst_Shr
 - Opioid_Drug_Flag
 - Opioid_LA_Drug_Flag
 - Antbtc_Drug_Flag
 - Antpsyct_Drug_Flag
 - data_year

FIRST 3 ROWS:


,Prscrbr_Geo_Lvl,Prscrbr_Geo_Cd,Prscrbr_Geo_Desc,Brnd_Name,Gnrc_Name,Tot_Prscrbrs,Tot_Clms,Tot_30day_Fills,Tot_Drug_Cst,Tot_Benes,...,GE65_Tot_Drug_Cst,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes,LIS_Bene_Cst_Shr,NonLIS_Bene_Cst_Shr,Opioid_Drug_Flag,Opioid_LA_Drug_Flag,Antbtc_Drug_Flag,Antpsyct_Drug_Flag,data_year
0,National,NaN,National,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1178,3708,6598.5,102280.76,1313.0,...,79992.58,NaN,1052.0,4167.66,22903.26,N,N,N,N,2021
1,National,NaN,National,1st Tier Unifine Pentips Plus,"Pen Needle, Diabetic",1604,4564,7905.6,131927.33,1766.0,...,106683.88,NaN,1450.0,4415.22,20335.83,N,N,N,N,2021
2,National,NaN,National,Abacavir,Abacavir Sulfate,3560,30534,37402.4,7039482.37,4252.0,...,3547013.16,NaN,2141.0,21278.04,186025.79,N,N,N,N,2021



CMS PART D — 2022

COLUMNS:
 - Prscrbr_Geo_Lvl
 - Prscrbr_Geo_Cd
 - Prscrbr_Geo_Desc
 - Brnd_Name
 - Gnrc_Name
 - Tot_Prscrbrs
 - Tot_Clms
 - Tot_30day_Fills
 - Tot_Drug_Cst
 - Tot_Benes
 - GE65_Sprsn_Flag
 - GE65_Tot_Clms
 - GE65_Tot_30day_Fills
 - GE65_Tot_Drug_Cst
 - GE65_Bene_Sprsn_Flag
 - GE65_Tot_Benes
 - LIS_Bene_Cst_Shr
 - NonLIS_Bene_Cst_Shr
 - Opioid_Drug_Flag
 - Opioid_LA_Drug_Flag
 - Antbtc_Drug_Flag
 - Antpsyct_Drug_Flag
 - data_year

FIRST 3 ROWS:


,Prscrbr_Geo_Lvl,Prscrbr_Geo_Cd,Prscrbr_Geo_Desc,Brnd_Name,Gnrc_Name,Tot_Prscrbrs,Tot_Clms,Tot_30day_Fills,Tot_Drug_Cst,Tot_Benes,...,GE65_Tot_Drug_Cst,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes,LIS_Bene_Cst_Shr,NonLIS_Bene_Cst_Shr,Opioid_Drug_Flag,Opioid_LA_Drug_Flag,Antbtc_Drug_Flag,Antpsyct_Drug_Flag,data_year
0,National,NaN,National,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1061,2501,4473.6,70039.61,1147.0,...,56669.52,NaN,933.0,2459.79,13488.17,N,N,N,N,2022
1,National,NaN,National,1st Tier Unifine Pentips Plus,"Pen Needle, Diabetic",1372,3846,6524.3,114601.54,1474.0,...,91959.96,NaN,1201.0,3363.09,15703.80,N,N,N,N,2022
2,National,NaN,National,Abacavir,Abacavir Sulfate,3161,24329,30526.9,6950841.50,3453.0,...,3686774.74,NaN,1888.0,14365.22,181969.45,N,N,N,N,2022



CMS PART D — 2023

COLUMNS:
 - Prscrbr_Geo_Lvl
 - Prscrbr_Geo_Cd
 - Prscrbr_Geo_Desc
 - Brnd_Name
 - Gnrc_Name
 - Tot_Prscrbrs
 - Tot_Clms
 - Tot_30day_Fills
 - Tot_Drug_Cst
 - Tot_Benes
 - GE65_Sprsn_Flag
 - GE65_Tot_Clms
 - GE65_Tot_30day_Fills
 - GE65_Tot_Drug_Cst
 - GE65_Bene_Sprsn_Flag
 - GE65_Tot_Benes
 - LIS_Bene_Cst_Shr
 - NonLIS_Bene_Cst_Shr
 - Opioid_Drug_Flag
 - Opioid_LA_Drug_Flag
 - Antbtc_Drug_Flag
 - Antpsyct_Drug_Flag
 - data_year

FIRST 3 ROWS:


,Prscrbr_Geo_Lvl,Prscrbr_Geo_Cd,Prscrbr_Geo_Desc,Brnd_Name,Gnrc_Name,Tot_Prscrbrs,Tot_Clms,Tot_30day_Fills,Tot_Drug_Cst,Tot_Benes,...,GE65_Tot_Drug_Cst,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes,LIS_Bene_Cst_Shr,NonLIS_Bene_Cst_Shr,Opioid_Drug_Flag,Opioid_LA_Drug_Flag,Antbtc_Drug_Flag,Antpsyct_Drug_Flag,data_year
0,National,NaN,National,1st Tier Unifine Pentips,"Pen Needle, Diabetic",691,1613,2874.2,44355.04,699.0,...,35803.14,NaN,565.0,992.77,7093.55,N,N,N,N,2023
1,National,NaN,National,1st Tier Unifine Pentips Plus,"Pen Needle, Diabetic",1167,3269,5592.0,97951.18,1267.0,...,76544.14,NaN,1035.0,1899.59,10392.03,N,N,N,N,2023
2,National,NaN,National,Abacavir,Abacavir Sulfate,2617,19634,25152.1,5290175.88,2807.0,...,3042569.00,NaN,1662.0,10643.29,152140.95,N,N,N,N,2023



CMS PART D — 2024

COLUMNS:
 - Prscrbr_Geo_Lvl
 - Prscrbr_Geo_Cd
 - Prscrbr_Geo_Desc
 - Brnd_Name
 - Gnrc_Name
 - Tot_Prscrbrs
 - Tot_Clms
 - Tot_30day_Fills
 - Tot_Drug_Cst
 - Tot_Benes
 - GE65_Sprsn_Flag
 - GE65_Tot_Clms
 - GE65_Tot_30day_Fills
 - GE65_Tot_Drug_Cst
 - GE65_Bene_Sprsn_Flag
 - GE65_Tot_Benes
 - LIS_Bene_Cst_Shr
 - NonLIS_Bene_Cst_Shr
 - Opioid_Drug_Flag
 - Opioid_LA_Drug_Flag
 - Antbtc_Drug_Flag
 - Antpsyct_Drug_Flag
 - data_year

FIRST 3 ROWS:


,Prscrbr_Geo_Lvl,Prscrbr_Geo_Cd,Prscrbr_Geo_Desc,Brnd_Name,Gnrc_Name,Tot_Prscrbrs,Tot_Clms,Tot_30day_Fills,Tot_Drug_Cst,Tot_Benes,...,GE65_Tot_Drug_Cst,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes,LIS_Bene_Cst_Shr,NonLIS_Bene_Cst_Shr,Opioid_Drug_Flag,Opioid_LA_Drug_Flag,Antbtc_Drug_Flag,Antpsyct_Drug_Flag,data_year
0,National,NaN,National,1st Tier Unifine Pentips,"Pen Needle, Diabetic",209,378,695.6,10454.60,213.0,...,8336.36,NaN,174.0,140.44,3053.27,N,N,N,N,2024
1,National,NaN,National,1st Tier Unifine Pentips Plus,"Pen Needle, Diabetic",442,965,1588.4,26074.56,451.0,...,20918.75,NaN,360.0,686.07,5152.25,N,N,N,N,2024
2,National,NaN,National,Abacavir,Abacavir Sulfate,2277,16298,21345.5,3054440.46,2349.0,...,1994516.36,NaN,1491.0,6378.08,106462.22,N,N,N,N,2024


In [9]:
# ============================================================
# PHARMAOPT — NATIONAL CMS DEMAND TABLES
# ROBUST ACROSS CMS YEAR-SPECIFIC COLUMN NAMES
# ============================================================

import re

cms_national = {}

print("BUILDING NATIONAL CMS DEMAND TABLES")
print("=" * 75)


def find_column(df, target_normalized):
    """
    Match CMS columns while ignoring underscores,
    spaces, capitalization, and punctuation.
    """

    for col in df.columns:

        normalized = re.sub(
            r"[^a-z0-9]",
            "",
            col.lower()
        )

        if normalized == target_normalized:
            return col

    return None


for year, df_original in cms_data.items():

    # Work on a copy
    df = df_original.copy()

    # --------------------------------------------------------
    # FIND YEAR-SPECIFIC COLUMN NAMES
    # --------------------------------------------------------

    geo_col = find_column(
        df,
        "prscrbrgeolvl"
    )

    brand_col = find_column(
        df,
        "brndname"
    )

    generic_col = find_column(
        df,
        "gnrcname"
    )

    claims_col = find_column(
        df,
        "totclms"
    )

    fills_col = find_column(
        df,
        "tot30dayfills"
    )

    cost_col = find_column(
        df,
        "totdrugcst"
    )


    column_map = {
        "Geography": geo_col,
        "Brand": brand_col,
        "Generic": generic_col,
        "Claims": claims_col,
        "30-Day Fills": fills_col,
        "Drug Cost": cost_col
    }


    print(f"\n{year} COLUMN MAPPING")
    print("-" * 50)

    for label, column in column_map.items():
        print(f"{label:<15} → {column}")


    # --------------------------------------------------------
    # MAKE SURE EVERYTHING WAS FOUND
    # --------------------------------------------------------

    missing = [
        label
        for label, column
        in column_map.items()
        if column is None
    ]

    if missing:

        raise ValueError(
            f"{year}: could not identify "
            f"these fields: {missing}"
        )


    # --------------------------------------------------------
    # STANDARDIZE COLUMN NAMES
    # --------------------------------------------------------

    df = df.rename(
        columns={
            geo_col: "Prscrbr_Geo_Lvl",
            brand_col: "Brnd_Name",
            generic_col: "Gnrc_Name",
            claims_col: "Tot_Clms",
            fills_col: "Tot30_Day_Fills",
            cost_col: "Tot_Drug_Cst"
        }
    )


    # --------------------------------------------------------
    # NATIONAL ROWS ONLY
    # --------------------------------------------------------

    national = df[
        df["Prscrbr_Geo_Lvl"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "national"
    ].copy()


    # --------------------------------------------------------
    # NUMERIC CONVERSION
    # --------------------------------------------------------

    numeric_columns = [
        "Tot_Clms",
        "Tot30_Day_Fills",
        "Tot_Drug_Cst"
    ]

    for col in numeric_columns:

        national[col] = pd.to_numeric(
            national[col],
            errors="coerce"
        )


    cms_national[year] = national


    print(
        f"\nNational drug rows: "
        f"{len(national):,}"
    )


print("\n" + "=" * 75)
print("ALL CMS YEARS STANDARDIZED SUCCESSFULLY")
print("=" * 75)

BUILDING NATIONAL CMS DEMAND TABLES

2021 COLUMN MAPPING
--------------------------------------------------
Geography       → Prscrbr_Geo_Lvl
Brand           → Brnd_Name
Generic         → Gnrc_Name
Claims          → Tot_Clms
30-Day Fills    → Tot_30day_Fills
Drug Cost       → Tot_Drug_Cst

National drug rows: 3,567

2022 COLUMN MAPPING
--------------------------------------------------
Geography       → Prscrbr_Geo_Lvl
Brand           → Brnd_Name
Generic         → Gnrc_Name
Claims          → Tot_Clms
30-Day Fills    → Tot_30day_Fills
Drug Cost       → Tot_Drug_Cst

National drug rows: 3,584

2023 COLUMN MAPPING
--------------------------------------------------
Geography       → Prscrbr_Geo_Lvl
Brand           → Brnd_Name
Generic         → Gnrc_Name
Claims          → Tot_Clms
30-Day Fills    → Tot_30day_Fills
Drug Cost       → Tot_Drug_Cst

National drug rows: 3,607

2024 COLUMN MAPPING
--------------------------------------------------
Geography       → Prscrbr_Geo_Lvl
Brand          

In [10]:
# ============================================================
# PHARMAOPT — CONSERVATIVE DRUG NAME NORMALIZATION
# ============================================================

import re
import unicodedata


def normalize_drug_name(value):

    if pd.isna(value):
        return None

    value = str(value)

    # Unicode normalization
    value = unicodedata.normalize(
        "NFKD",
        value
    )

    # Lowercase
    value = value.lower()

    # Replace punctuation with spaces
    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )

    # Collapse repeated spaces
    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


for year in cms_national:

    cms_national[year]["generic_normalized"] = (
        cms_national[year]["Gnrc_Name"]
        .apply(normalize_drug_name)
    )

    cms_national[year]["brand_normalized"] = (
        cms_national[year]["Brnd_Name"]
        .apply(normalize_drug_name)
    )


print("Drug-name normalization complete.")

display(
    cms_national[2024][
        [
            "Gnrc_Name",
            "generic_normalized",
            "Brnd_Name",
            "brand_normalized"
        ]
    ].head(10)
)

Drug-name normalization complete.


,Gnrc_Name,generic_normalized,Brnd_Name,brand_normalized
0,"Pen Needle, Diabetic",pen needle diabetic,1st Tier Unifine Pentips,1st tier unifine pentips
1,"Pen Needle, Diabetic",pen needle diabetic,1st Tier Unifine Pentips Plus,1st tier unifine pentips plus
2,Abacavir Sulfate,abacavir sulfate,Abacavir,abacavir
3,Abacavir Sulfate/Lamivudine,abacavir sulfate lamivudine,Abacavir-Lamivudine,abacavir lamivudine
4,Aripiprazole,aripiprazole,Abilify,abilify
5,Aripiprazole,aripiprazole,Abilify Asimtufii,abilify asimtufii
6,Aripiprazole,aripiprazole,Abilify Maintena,abilify maintena
7,Aripiprazole,aripiprazole,Abilify Mycite,abilify mycite
8,Abiraterone Acetate,abiraterone acetate,Abiraterone Acetate,abiraterone acetate
9,Paclitaxel Protein-Bound,paclitaxel protein bound,Abraxane,abraxane


In [11]:
# ============================================================
# PHARMAOPT — GENERIC-LEVEL DEMAND
# ============================================================

generic_demand_frames = []

for year, df in cms_national.items():

    generic = (
        df[
            df["generic_normalized"].notna()
        ]
        .groupby(
            "generic_normalized",
            as_index=False
        )
        .agg(

            cms_total_claims=(
                "Tot_Clms",
                "sum"
            ),

            cms_30day_fills=(
                "Tot30_Day_Fills",
                "sum"
            ),

            cms_total_drug_cost=(
                "Tot_Drug_Cst",
                "sum"
            ),

            cms_brand_count=(
                "Brnd_Name",
                "nunique"
            )
        )
    )

    generic["cms_data_year"] = year

    # This year's utilization predicts NEXT year's shortage
    generic["prediction_year"] = year + 1

    generic_demand_frames.append(
        generic
    )


cms_generic_demand = pd.concat(
    generic_demand_frames,
    ignore_index=True
)


print("CMS GENERIC DEMAND TABLE")
print("=" * 75)

print(
    "Rows:",
    f"{len(cms_generic_demand):,}"
)

print(
    "Unique normalized generics:",
    f"{cms_generic_demand['generic_normalized'].nunique():,}"
)

display(
    cms_generic_demand.head(10)
)

CMS GENERIC DEMAND TABLE
Rows: 7,737
Unique normalized generics: 2,147


,generic_normalized,cms_total_claims,cms_30day_fills,cms_total_drug_cost,cms_brand_count,cms_data_year,prediction_year
0,0 9 sodium chloride,248597,251845.1,5.432729e+06,1,2021,2022
1,aa 5 calcium lytes dext 20,107,107.0,2.891125e+04,1,2021,2022
2,aa 5 d15w electrolytes,41,41.0,1.921016e+04,1,2021,2022
3,abacavir dolutegravir lamivudi,201830,212766.3,6.971825e+08,1,2021,2022
4,abacavir lamivudine zidovudine,880,941.0,1.430763e+06,2,2021,2022
5,abacavir sulfate,30816,37755.7,7.219637e+06,2,2021,2022
6,abacavir sulfate lamivudine,31141,39848.9,1.101490e+07,2,2021,2022
7,abaloparatide,58815,62961.5,1.378310e+08,1,2021,2022
8,abatacept,78178,87246.5,4.361922e+08,2,2021,2022
9,abatacept maltose,8524,8694.5,3.495268e+07,1,2021,2022


In [12]:
# ============================================================
# PHARMAOPT — DEMAND GROWTH FEATURES
# ============================================================

cms_generic_demand = (
    cms_generic_demand
    .sort_values(
        [
            "generic_normalized",
            "cms_data_year"
        ]
    )
    .reset_index(drop=True)
)


# Previous-year values
cms_generic_demand["previous_claims"] = (
    cms_generic_demand
    .groupby("generic_normalized")
    ["cms_total_claims"]
    .shift(1)
)

cms_generic_demand["previous_fills"] = (
    cms_generic_demand
    .groupby("generic_normalized")
    ["cms_30day_fills"]
    .shift(1)
)


# Growth rates
cms_generic_demand["cms_claim_growth"] = (
    (
        cms_generic_demand["cms_total_claims"]
        -
        cms_generic_demand["previous_claims"]
    )
    /
    cms_generic_demand["previous_claims"]
)


cms_generic_demand["cms_fill_growth"] = (
    (
        cms_generic_demand["cms_30day_fills"]
        -
        cms_generic_demand["previous_fills"]
    )
    /
    cms_generic_demand["previous_fills"]
)


# Prevent infinite values from division edge cases
cms_generic_demand[
    [
        "cms_claim_growth",
        "cms_fill_growth"
    ]
] = (
    cms_generic_demand[
        [
            "cms_claim_growth",
            "cms_fill_growth"
        ]
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# Log transforms for very large utilization values
cms_generic_demand["log_cms_claims"] = (
    np.log1p(
        cms_generic_demand[
            "cms_total_claims"
        ]
    )
)

cms_generic_demand["log_cms_fills"] = (
    np.log1p(
        cms_generic_demand[
            "cms_30day_fills"
        ]
    )
)

cms_generic_demand["log_cms_cost"] = (
    np.log1p(
        cms_generic_demand[
            "cms_total_drug_cost"
        ]
    )
)


print("DEMAND FEATURES CREATED")
print("=" * 75)

display(
    cms_generic_demand[
        [
            "generic_normalized",
            "cms_data_year",
            "prediction_year",
            "cms_total_claims",
            "cms_30day_fills",
            "cms_claim_growth",
            "cms_fill_growth"
        ]
    ].head(20)
)

DEMAND FEATURES CREATED


,generic_normalized,cms_data_year,prediction_year,cms_total_claims,cms_30day_fills,cms_claim_growth,cms_fill_growth
0,0 9 sodium chloride,2021,2022,248597,251845.1,NaN,NaN
1,0 9 sodium chloride,2022,2023,259126,262867.4,0.042354,0.043766
2,0 9 sodium chloride,2023,2024,303702,310872.7,0.172024,0.182622
3,0 9 sodium chloride,2024,2025,345300,351967.5,0.136970,0.132192
4,aa 4 25 calcium lytes dex 10,2022,2023,12,12.0,NaN,NaN
5,aa 4 25 no 1 d10w lytes no 39,2024,2025,17,17.0,NaN,NaN
6,aa 4 25 no 1 d5w lytes no 39,2024,2025,15,15.0,NaN,NaN
7,aa 4 25 no 1 dextrose 5,2024,2025,72,72.0,NaN,NaN
8,aa 5 calcium lytes dext 20,2021,2022,107,107.0,NaN,NaN
9,aa 5 calcium lytes dext 20,2022,2023,64,64.0,-0.401869,-0.401869


In [13]:
# ============================================================
# PHARMAOPT — LOAD FDA PRODUCT NAMES
# ============================================================

import json
import zipfile

ndc_zip = (
    PROJECT_DIR /
    "drug-ndc-0001-of-0001.json.zip"
)


with zipfile.ZipFile(
    ndc_zip,
    "r"
) as z:

    json_member = z.namelist()[0]

    with z.open(json_member) as f:
        ndc_json = json.load(f)


fda_products = pd.json_normalize(
    ndc_json["results"]
)


fda_names = fda_products[
    [
        "product_id",
        "product_ndc",
        "generic_name",
        "brand_name",
        "labeler_name"
    ]
].copy()


fda_names["generic_normalized"] = (
    fda_names["generic_name"]
    .apply(normalize_drug_name)
)


print("FDA products loaded:")
print(f"{len(fda_names):,}")

print(
    "Unique FDA normalized generic names:",
    f"{fda_names['generic_normalized'].nunique():,}"
)

FDA products loaded:
138,217
Unique FDA normalized generic names: 18,819


In [14]:
# ============================================================
# PHARMAOPT — CMS ↔ FDA GENERIC NAME LINKAGE AUDIT
# ============================================================

fda_generic_set = set(
    fda_names[
        "generic_normalized"
    ].dropna()
)


cms_generic_demand["fda_generic_match"] = (
    cms_generic_demand[
        "generic_normalized"
    ]
    .isin(fda_generic_set)
)


print("CMS → FDA GENERIC LINKAGE")
print("=" * 75)


for year in sorted(
    cms_generic_demand[
        "cms_data_year"
    ].unique()
):

    temp = cms_generic_demand[
        cms_generic_demand[
            "cms_data_year"
        ] == year
    ]

    matched = int(
        temp["fda_generic_match"].sum()
    )

    total = len(temp)

    # Demand-weighted match coverage
    total_claims = (
        temp["cms_total_claims"].sum()
    )

    matched_claims = (
        temp.loc[
            temp["fda_generic_match"],
            "cms_total_claims"
        ].sum()
    )

    row_match_rate = (
        matched / total * 100
        if total else 0
    )

    claim_match_rate = (
        matched_claims /
        total_claims *
        100
        if total_claims else 0
    )


    print(
        f"{year}: "
        f"{matched:,}/{total:,} generic names matched "
        f"({row_match_rate:.2f}%) | "
        f"{claim_match_rate:.2f}% of claims covered"
    )

CMS → FDA GENERIC LINKAGE
2021: 1,164/1,884 generic names matched (61.78%) | 87.02% of claims covered
2022: 1,188/1,920 generic names matched (61.88%) | 87.09% of claims covered
2023: 1,222/1,945 generic names matched (62.83%) | 86.85% of claims covered
2024: 1,249/1,988 generic names matched (62.83%) | 87.19% of claims covered


In [15]:
# ============================================================
# SAVE PHARMAOPT CMS DEMAND FEATURES
# ============================================================

output_columns = [

    "generic_normalized",

    "cms_data_year",
    "prediction_year",

    "cms_total_claims",
    "cms_30day_fills",
    "cms_total_drug_cost",
    "cms_brand_count",

    "cms_claim_growth",
    "cms_fill_growth",

    "log_cms_claims",
    "log_cms_fills",
    "log_cms_cost",

    "fda_generic_match"
]


cms_feature_file = (
    PROJECT_DIR /
    "cms_generic_demand_features.csv"
)


cms_generic_demand[
    output_columns
].to_csv(
    cms_feature_file,
    index=False
)


print("SAVED:")
print(cms_feature_file)

print(
    "\nRows:",
    f"{len(cms_generic_demand):,}"
)

SAVED:
C:\Users\jpiza\PharmaOpt\cms_generic_demand_features.csv

Rows: 7,737


In [18]:
# ============================================================
# PHARMAOPT — FIND + LOAD HISTORICAL CMS DATA
# 2019 AND 2020
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd
import shutil
import re

downloads = Path.home() / "Downloads"
project = Path.cwd()

target_years = [2019, 2020]

found_members = {}

# Search both Downloads and PharmaOpt
search_locations = [
    downloads,
    project
]

for location in search_locations:

    for zip_path in location.glob("*.zip"):

        # Only inspect CMS Part D ZIPs
        if "Medicare Part D Prescribers" not in zip_path.name:
            continue

        try:

            with zipfile.ZipFile(zip_path, "r") as z:

                for member in z.namelist():

                    if not member.lower().endswith(".csv"):
                        continue

                    for year in target_years:

                        # Look for year folder OR DY19 / DY20
                        short_year = str(year)[-2:]

                        if (
                            f"/{year}/" in member
                            or f"DY{short_year}" in member.upper()
                        ):

                            found_members[year] = {
                                "zip_path": zip_path,
                                "member": member
                            }

        except zipfile.BadZipFile:
            pass


print("HISTORICAL CMS FILES FOUND")
print("=" * 90)

for year in target_years:

    if year not in found_members:

        print(f"{year}: NOT FOUND")

    else:

        info = found_members[year]

        print(f"\n{year}")
        print("ZIP:", info["zip_path"].name)
        print("CSV:", info["member"])


missing_years = [
    year
    for year in target_years
    if year not in found_members
]

if missing_years:

    raise FileNotFoundError(
        f"Could not find CMS data for: {missing_years}"
    )


# ============================================================
# COPY RAW ZIP(S) INTO PHARMAOPT
# ============================================================

for year, info in found_members.items():

    source = info["zip_path"]

    # Only copy if currently in Downloads
    if source.parent == downloads:

        destination = (
            project /
            f"cms_partd_historical_{year}.zip"
        )

        if not destination.exists():

            shutil.copy2(
                source,
                destination
            )

        # Point to permanent PharmaOpt copy
        found_members[year]["zip_path"] = destination


# ============================================================
# LOAD 2019 AND 2020
# ============================================================

for year in target_years:

    info = found_members[year]

    with zipfile.ZipFile(
        info["zip_path"],
        "r"
    ) as z:

        with z.open(
            info["member"]
        ) as f:

            cms_data[year] = pd.read_csv(
                f,
                low_memory=False
            )

    cms_data[year]["data_year"] = year

    print(
        f"\nLoaded {year}: "
        f"{len(cms_data[year]):,} rows × "
        f"{len(cms_data[year].columns):,} columns"
    )


print("\n" + "=" * 90)
print("CMS YEARS CURRENTLY AVAILABLE")
print("=" * 90)

for year in sorted(cms_data):

    print(
        f"{year}: "
        f"{len(cms_data[year]):,} rows"
    )

HISTORICAL CMS FILES FOUND

2019
ZIP: Medicare Part D Prescribers - by Geography and Drug (2).zip
CSV: Medicare Part D Prescribers - by Geography and Drug/2019/MUP_DPR_RY21_P04_V10_DY19_Geo.csv

2020
ZIP: Medicare Part D Prescribers - by Geography and Drug (2).zip
CSV: Medicare Part D Prescribers - by Geography and Drug/2020/MUP_DPR_RY22_P04_V10_DY20_Geo.csv

Loaded 2019: 114,645 rows × 23 columns

Loaded 2020: 115,410 rows × 23 columns

CMS YEARS CURRENTLY AVAILABLE
2019: 114,645 rows
2020: 115,410 rows
2021: 114,891 rows
2022: 115,396 rows
2023: 115,936 rows
2024: 117,661 rows


In [19]:

# ============================================================
# PHARMAOPT — STANDARDIZE CMS 2019–2024
# ============================================================

cms_national = {}


def find_column(df, target_normalized):

    for col in df.columns:

        normalized = re.sub(
            r"[^a-z0-9]",
            "",
            col.lower()
        )

        if normalized == target_normalized:
            return col

    return None


for year in sorted(cms_data):

    df = cms_data[year].copy()

    geo_col = find_column(df, "prscrbrgeolvl")
    brand_col = find_column(df, "brndname")
    generic_col = find_column(df, "gnrcname")
    claims_col = find_column(df, "totclms")
    fills_col = find_column(df, "tot30dayfills")
    cost_col = find_column(df, "totdrugcst")

    mapping = {
        "Geography": geo_col,
        "Brand": brand_col,
        "Generic": generic_col,
        "Claims": claims_col,
        "30-Day Fills": fills_col,
        "Drug Cost": cost_col
    }

    missing = [
        label
        for label, column in mapping.items()
        if column is None
    ]

    if missing:
        raise ValueError(
            f"{year}: missing fields {missing}"
        )

    df = df.rename(
        columns={
            geo_col: "Prscrbr_Geo_Lvl",
            brand_col: "Brnd_Name",
            generic_col: "Gnrc_Name",
            claims_col: "Tot_Clms",
            fills_col: "Tot30_Day_Fills",
            cost_col: "Tot_Drug_Cst"
        }
    )

    national = df[
        df["Prscrbr_Geo_Lvl"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "national"
    ].copy()

    for col in [
        "Tot_Clms",
        "Tot30_Day_Fills",
        "Tot_Drug_Cst"
    ]:

        national[col] = pd.to_numeric(
            national[col],
            errors="coerce"
        )

    cms_national[year] = national

    print(
        f"{year}: "
        f"{len(national):,} national drug rows"
    )


print("\nALL SIX YEARS STANDARDIZED")


2019: 3,544 national drug rows
2020: 3,582 national drug rows
2021: 3,567 national drug rows
2022: 3,584 national drug rows
2023: 3,607 national drug rows
2024: 3,632 national drug rows

ALL SIX YEARS STANDARDIZED


In [20]:
# ============================================================
# PHARMAOPT — CMS HISTORICAL DEMAND FEATURES
# ============================================================

generic_frames = []


for year in sorted(cms_national):

    df = cms_national[year].copy()

    df["generic_normalized"] = (
        df["Gnrc_Name"]
        .apply(normalize_drug_name)
    )

    df["brand_normalized"] = (
        df["Brnd_Name"]
        .apply(normalize_drug_name)
    )

    generic = (
        df[
            df["generic_normalized"].notna()
        ]
        .groupby(
            "generic_normalized",
            as_index=False
        )
        .agg(

            cms_total_claims=(
                "Tot_Clms",
                "sum"
            ),

            cms_30day_fills=(
                "Tot30_Day_Fills",
                "sum"
            ),

            cms_total_drug_cost=(
                "Tot_Drug_Cst",
                "sum"
            ),

            cms_brand_count=(
                "Brnd_Name",
                "nunique"
            )
        )
    )

    generic["cms_data_year"] = year

    generic_frames.append(generic)


cms_history = pd.concat(
    generic_frames,
    ignore_index=True
)


cms_history = (
    cms_history
    .sort_values(
        [
            "generic_normalized",
            "cms_data_year"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# YEAR-OVER-YEAR FEATURES
# ------------------------------------------------------------

cms_history["previous_claims"] = (
    cms_history
    .groupby("generic_normalized")
    ["cms_total_claims"]
    .shift(1)
)

cms_history["previous_fills"] = (
    cms_history
    .groupby("generic_normalized")
    ["cms_30day_fills"]
    .shift(1)
)


cms_history["cms_claim_growth"] = (
    (
        cms_history["cms_total_claims"]
        -
        cms_history["previous_claims"]
    )
    /
    cms_history["previous_claims"]
)


cms_history["cms_fill_growth"] = (
    (
        cms_history["cms_30day_fills"]
        -
        cms_history["previous_fills"]
    )
    /
    cms_history["previous_fills"]
)


cms_history[
    [
        "cms_claim_growth",
        "cms_fill_growth"
    ]
] = (
    cms_history[
        [
            "cms_claim_growth",
            "cms_fill_growth"
        ]
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# ------------------------------------------------------------
# LOG FEATURES
# ------------------------------------------------------------

cms_history["log_cms_claims"] = (
    np.log1p(
        cms_history["cms_total_claims"]
    )
)

cms_history["log_cms_fills"] = (
    np.log1p(
        cms_history["cms_30day_fills"]
    )
)

cms_history["log_cms_cost"] = (
    np.log1p(
        cms_history["cms_total_drug_cost"]
    )
)


print(
    "Historical CMS feature rows:",
    f"{len(cms_history):,}"
)

print()

print(
    cms_history["cms_data_year"]
    .value_counts()
    .sort_index()
)

Historical CMS feature rows: 11,457

cms_data_year
2019    1847
2020    1873
2021    1884
2022    1920
2023    1945
2024    1988
Name: count, dtype: int64


In [21]:
# ============================================================
# PHARMAOPT — STRICT TEMPORAL ALIGNMENT
# ============================================================

strict_year_map = {

    2019: 2022,
    2020: 2023,
    2021: 2024,
    2022: 2025

}


cms_strict = cms_history[
    cms_history["cms_data_year"]
    .isin(strict_year_map)
].copy()


cms_strict["prediction_year"] = (
    cms_strict["cms_data_year"]
    .map(strict_year_map)
)


print("STRICT CMS FORECAST ALIGNMENT")
print("=" * 60)

for cms_year, prediction_year in strict_year_map.items():

    rows = len(
        cms_strict[
            cms_strict["cms_data_year"]
            == cms_year
        ]
    )

    print(
        f"CMS {cms_year} "
        f"→ shortage year {prediction_year} "
        f"| {rows:,} generic drugs"
    )

STRICT CMS FORECAST ALIGNMENT
CMS 2019 → shortage year 2022 | 1,847 generic drugs
CMS 2020 → shortage year 2023 | 1,873 generic drugs
CMS 2021 → shortage year 2024 | 1,884 generic drugs
CMS 2022 → shortage year 2025 | 1,920 generic drugs


In [22]:
# ============================================================
# SAVE STRICT CMS FEATURES
# ============================================================

strict_columns = [

    "generic_normalized",

    "cms_data_year",
    "prediction_year",

    "cms_total_claims",
    "cms_30day_fills",
    "cms_total_drug_cost",
    "cms_brand_count",

    "cms_claim_growth",
    "cms_fill_growth",

    "log_cms_claims",
    "log_cms_fills",
    "log_cms_cost"
]


strict_file = (
    PROJECT_DIR /
    "cms_strict_demand_features.csv"
)


cms_strict[
    strict_columns
].to_csv(
    strict_file,
    index=False
)


print("STRICT CMS DATASET SAVED")
print("=" * 60)

print(strict_file)

print(
    "\nRows:",
    f"{len(cms_strict):,}"
)

STRICT CMS DATASET SAVED
C:\Users\jpiza\PharmaOpt\cms_strict_demand_features.csv

Rows: 7,524
